In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway_2007 = os.path.join(pathway_temp, "brauer2007chimpanzees")
original_data_pathway = os.path.join(pathway_2007, "original_data")
pathway_2008 = os.path.join(pathway_temp, "brauer2008chimpanzees")
original_data_pathway = os.path.join(pathway_2008, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2007_2008_kummernoise-basic-data.sav")
complete_path_2 = os.path.join(original_data_pathway, "Braeuer_2007_2008_kummernoise-basic-data2.sav")


out_pathway_2007 = os.path.join(pathway_2007, "standardized_data")
if not os.path.exists(out_pathway_2007):
    os.makedirs(out_pathway_2007)

out_pathway_2008 = os.path.join(pathway_2008, "standardized_data")
if not os.path.exists(out_pathway_2008):
    os.makedirs(out_pathway_2008)

In [2]:

import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
experiment_import = [[df1, 'basic_data', '1'],
                    [df2, 'basic_data2','2']]
for x,y,k in experiment_import:
    x['experiment_name']=y
    x['experiment']=k



In [3]:
publication=[]
for index, row in df1.iterrows():
    if "noise" in str(row['cond_sid']):
        year = str('unpublished')
        publication.append(year)
    else:
        publication.append('2007')
#print(publication)
df1 = df1.assign(publication=publication)


In [4]:
publication2=[]
for index, row in df2.iterrows():
    if "noise" in str(row['cond_sid']):
        year = str('2008')
        publication2.append(year)
    else:
        publication2.append('2007')
# print(publication2)
df2 = df2.assign(publication=publication2)

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
data_frames=[df1, df2]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s)
    # x['study_id']="brauer2007chimpanzees_brauer2008chimpanzees"
    x['subject'] = x['subject'].str.rstrip()
    x['against'] = x['against'].str.rstrip()
    x = x.rename(columns={"subject": "ape"})
    x = x.rename(columns={"against": "ape_2"})
    data_frames[index]=x
new_df=data_frames[0]

fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['ape_2'].replace(x, y, inplace=True)

fulldf['dyad']=fulldf.ape.str.cat(fulldf.ape_2, sep='_')
# fulldf.columns

In [6]:
fulldf['role']='focal_participant'
fulldf['role_2']='against'



In [7]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [8]:
fulldf = fulldf.rename(columns={"species_y": "species",
                                "date":"date_temp"})
# df.columns
# fulldf['date_temp'].unique()
fulldf['date_temp'].replace('', np.nan, inplace=True)

In [9]:
# temp_var = ''
# out_list = []
# for index, row in fulldf.iterrows():
#     if not pd.isna(row['date_temp']):
#         temp_var = row['date_temp']
#     out_list.append(temp_var)
# fulldf = fulldf.assign(date=out_list)

fulldf['date'] = fulldf['date_temp'].astype(str).str.pad(8, 'left', '0')
fulldf['date'].replace('00000000', np.nan, inplace=True)
fulldf['date'].replace('00000nan', np.nan, inplace=True)
# fulldf['date'].unique()

In [10]:
temp_var = ''
out_list = []
for index, row in fulldf.iterrows():
    if not pd.isna(row['date']):
        temp_var = row['date']
    out_list.append(temp_var)
fulldf = fulldf.assign(date_temp=out_list)

In [11]:
fulldf['month'] = fulldf['date_temp'].str.slice(0,2)
fulldf['day'] = fulldf['date_temp'].str.slice(2,4)
fulldf['year'] = fulldf['date_temp'].str.slice(4,8)


In [12]:
fulldf.dropna(subset=['ape'], inplace=True)

In [13]:
out = [(i+1) for l in range(56) for i in range(12)]

fulldf = fulldf.assign(trial=out)

fulldf['study_id']="brauer2007chimpanzees"

In [14]:
import re
replace_1= re.compile('( |-|\()') 
replacement_list = ['cond_sid', 'cond', 'app_what', 'con_inde']
for x in replacement_list:
    fulldf[x] = fulldf[x].str.replace(replace_1, '_')

replacement_2 = ['cond_sid','cond']
for x in replacement_2:
    fulldf[x] = fulldf[x].str.replace('__', '_', regex=False)
    fulldf[x] = fulldf[x].str.replace(')', '', regex=False)

In [15]:
fulldf.rename(columns={"ape": "participant", "ape_2":"participant_2",
                       "cond":"condition"}, inplace=True)

In [16]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
fulldf= fulldf.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    fulldf[x] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
    fulldf[x].replace('nan-nan-nan', np.nan, inplace=True)
    fulldf[x] = pd.to_datetime(fulldf[x])
    fulldf[y] = pd.to_datetime(fulldf[y])
    fulldf[k] = (fulldf[x] - fulldf[y]).dt.days//365

In [17]:
# 296
fulldf.at[296, 'app_side'] = 'right'
fulldf['app_side'].unique()

array(['left', 'not', 'right'], dtype=object)

In [18]:
fulldf=fulldf[['study_id', 'experiment','experiment_name', 'publication','year', 'month', 'day', 
         'participant','age_in_years',
        'sex','role','participant_2','age_in_years_2','sex_2','role_2', 'species', 'dyad', 'trial', 'pair',  'cond_sid', 'app_side', 'app_what',
       'app_time',  'condition', 'whath', 'whatv', 'whathn', 'whatvn',
       'con_inde' ]]


In [19]:
# exp1 = fulldf[fulldf['experiment'] == 'basic_data']
# exp1 = exp1.dropna(axis=1, how='all')

# exp2 = fulldf[fulldf['experiment'] == 'basic_data2']
# exp2 = exp2.dropna(axis=1, how='all')

In [20]:
# comp_out_path_stand = os.path.join(out_pathway_2007, 'brauer2007chimpanzees_brauer2008chimpanzees_exp1_standardized.csv')
# exp1.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

# comp_out_path_stand_1 = os.path.join(out_pathway_2007, 'brauer2007chimpanzees_brauer2008chimpanzees_exp2_standardized.csv')
# exp2.to_csv(comp_out_path_stand_1, encoding='utf-8-sig', index=False)


In [21]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway_2007, 'brauer2007chimpanzees_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway_2007, 'brauer2007chimpanzees_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

In [22]:
fulldf['study_id'].replace('brauer2007chimpanzees', 'brauer2008chimpanzees', inplace=True, regex=True)

In [23]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway_2008, 'brauer2008chimpanzees_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway_2008, 'brauer2008chimpanzees_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)